# Mutual Fund Analytics — Exploratory Data Analysis (EDA)
This notebook contains the complete visual and quantitative EDA pipeline for the Bluestock Mutual Fund Capstone project.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Setup directories
os.makedirs("reports/figures", exist_ok=True)
sns.set_theme(style="whitegrid")

# Load Datasets
folio_df = pd.read_csv("06_industry_folio_count.csv")
perf_df = pd.read_csv("07_scheme_performance.csv")
tx_df = pd.read_csv("08_investor_transactions.csv")
holdings_df = pd.read_csv("09_portfolio_holdings.csv")
benchmark_df = pd.read_csv("10_benchmark_indices.csv")

# Combine NAV datasets
nav_files = glob.glob("nav_*.csv")
if os.path.exists("hdfc_top100_live_nav.csv"):
    hdfc_df = pd.read_csv("hdfc_top100_live_nav.csv")
    hdfc_df["amfi_code"] = 125497
    nav_files.append("hdfc_top100_live_nav.csv")

nav_list = []
for file in nav_files:
    df = pd.read_csv(file)
    if "amfi_code" not in df.columns:
        df["amfi_code"] = 125497
    nav_list.append(df)

nav_df = pd.concat(nav_list, ignore_index=True)
nav_df["date"] = pd.to_datetime(nav_df["date"], errors="coerce")
nav_df["nav"] = pd.to_numeric(nav_df["nav"], errors="coerce")
nav_df = nav_df.dropna(subset=["date", "nav"]).sort_values(["amfi_code", "date"])

print("Datasets successfully loaded!")

## 1. NAV Trend Analysis
Tracking daily NAV trends across funds highlighting bull run and market correction periods.

In [ ]:
# 1. NAV Trend Analysis (Plotly)
fig = px.line(nav_df, x='date', y='nav', color='amfi_code', 
              title='Daily NAV Trend for Mutual Fund Schemes (2022–2026)')

fig.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.15,
              annotation_text="2023 Bull Run", annotation_position="top left")

fig.add_vrect(x0="2024-01-01", x1="2024-06-01", fillcolor="red", opacity=0.15,
              annotation_text="2024 Market Correction", annotation_position="top left")

fig.write_image("reports/figures/01_nav_trend_analysis.png")
fig.show()

## 2. AUM Growth Analysis
Comparing total AUM across major fund houses.

In [ ]:
# 2. AUM Growth Bar Chart
plt.figure(figsize=(10, 6))
sns.barplot(data=perf_df, x='fund_house', y='aum_crore', palette='Blues_r')
plt.title('AUM Concentration by Fund House')
plt.xticks(rotation=45)
plt.ylabel('AUM (₹ Crores)')
plt.tight_layout()
plt.savefig("reports/figures/02_aum_growth.png")
plt.show()

## 3. Monthly SIP Inflow Trend
Tracking total SIP inflows over time and annotating key historic highs.

In [ ]:
# 3. SIP Inflow Time-Series
monthly_sip = tx_df[tx_df['transaction_type'] == 'SIP'].groupby(
    pd.to_datetime(tx_df['transaction_date']).dt.to_period('M')
)['amount_inr'].sum().reset_index()
monthly_sip['transaction_date'] = monthly_sip['transaction_date'].dt.to_timestamp()

fig = px.line(monthly_sip, x='transaction_date', y='amount_inr', title='Monthly SIP Inflow Trend (2022–2025)')
fig.add_annotation(x='2025-12-01', y=31002000000, text="Dec 2025 ATH: ₹31,002 Cr",
                   showarrow=True, arrowhead=1)

fig.write_image("reports/figures/03_sip_inflow_trend.png")
fig.show()

## 4. Category Inflow Heatmap
Visualizing net inflow intensity across fund categories by month.

In [ ]:
# 4. Category Inflow Heatmap
cat_tx = tx_df.merge(perf_df[['amfi_code', 'category']], on='amfi_code', how='left')
cat_tx['month'] = pd.to_datetime(cat_tx['transaction_date']).dt.to_period('M')
heatmap_data = cat_tx.pivot_table(index='category', columns='month', values='amount_inr', aggfunc='sum').fillna(0)

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=False, fmt=".0f")
plt.title('Category-Wise Net Monthly Inflow Heatmap')
plt.xlabel('Month')
plt.ylabel('Fund Category')
plt.tight_layout()
plt.savefig("reports/figures/04_category_inflow_heatmap.png")
plt.show()

## 5. Investor Demographics
Analyzing age distributions, SIP investment patterns, and gender split.

In [ ]:
# 5. Investor Demographics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution pie chart
age_counts = tx_df['age_group'].value_counts()
axes[0].pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', startangle=140)
axes[0].set_title('Investor Age Group Distribution')

# SIP Amount Boxplot
sns.boxplot(data=tx_df[tx_df['transaction_type']=='SIP'], x='age_group', y='amount_inr', ax=axes[1])
axes[1].set_title('SIP Amount Distribution by Age Group')

# Gender Split
sns.countplot(data=tx_df, x='gender', ax=axes[2], palette='pastel')
axes[2].set_title('Gender Split')

plt.tight_layout()
plt.savefig("reports/figures/05_investor_demographics.png")
plt.show()

## 6. Geographic Distribution
Horizontal bar plot of SIP inflows by state and T30 vs B30 city tier pie chart.

In [ ]:
# 6. Geographic Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

state_sip = tx_df.groupby('state')['amount_inr'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=state_sip.values, y=state_sip.index, ax=axes[0], palette='viridis')
axes[0].set_title('Top 10 States by Total SIP Amount')

tier_counts = tx_df['city_tier'].value_counts()
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
axes[1].set_title('City Tier Distribution (T30 vs B30)')

plt.tight_layout()
plt.savefig("reports/figures/06_geographic_distribution.png")
plt.show()

## 7. Folio Count Growth
Tracking growth in mutual fund folios from 13.26 Cr to 26.12 Cr.

In [ ]:
# 7. Folio Count Growth
plt.figure(figsize=(10, 5))
sns.lineplot(data=folio_df, x='month', y='total_folios_crore', marker='o', color='purple')
plt.title('Industry Folio Count Growth (Jan 2022 – Dec 2025)')
plt.xticks(rotation=45)
plt.ylabel('Total Folios (Crore)')
plt.tight_layout()
plt.savefig("reports/figures/07_folio_count_growth.png")
plt.show()

## 8. Return Correlation Matrix
Pairwise return correlations between funds.

In [ ]:
# 8. NAV Return Correlation Matrix
nav_pivot = nav_df.pivot(index='date', columns='amfi_code', values='nav').pct_change().dropna()
corr = nav_pivot.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('NAV Daily Returns Correlation Matrix')
plt.tight_layout()
plt.savefig("reports/figures/08_nav_correlation_matrix.png")
plt.show()

## 9. Sector Allocation Donut Chart
Aggregated sector weight allocation across equity portfolios.

In [ ]:
# 9. Sector Allocation Donut Chart
sector_weights = holdings_df.groupby('sector')['weight_pct'].sum().sort_values(ascending=False).head(8)

plt.figure(figsize=(7, 7))
plt.pie(sector_weights, labels=sector_weights.index, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
centre_circle = plt.Circle((0,0),0.70,fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Top Sector Allocations across Portfolios')
plt.tight_layout()
plt.savefig("reports/figures/09_sector_allocation_donut.png")
plt.show()

## 10 Key EDA Findings

1. **NAV Trend Resilience**: Mutual fund NAVs showed steady recovery post early-2024 corrections as observed in *Figure 1: NAV Trend Analysis*.
2. **AUM Concentration**: A small group of premier fund houses dominate overall AUM as shown in *Figure 2: AUM Growth*.
3. **Record SIP Growth**: Monthly SIP inflows hit an all-time peak of ₹31,002 Cr in Dec 2025 as documented in *Figure 3: SIP Inflow Trend*.
4. **Category Preference**: Equity funds maintained the highest net monthly inflows compared to debt and hybrid categories in *Figure 4: Category Inflow Heatmap*.
5. **Young Investor Participation**: Investors aged 25–35 represent the highest share of active folios in *Figure 5: Investor Demographics*.
6. **SIP Ticket Sizes**: Middle-aged investors (36-50) maintain larger average SIP ticket sizes as highlighted in *Figure 5: Investor Demographics*.
7. **Geographic Inflows**: Top metro states contribute over 50% of overall mutual fund investment volumes in *Figure 6: Geographic Distribution*.
8. **B30 Expansion**: Beyond Top 30 (B30) cities show rapid growth accounting for ~30% total participation in *Figure 6: Geographic Distribution*.
9. **Folio Doubling**: Total industry folios nearly doubled from 13.26 Cr to 26.12 Cr over the 4-year period in *Figure 7: Folio Count Growth*.
10. **Sector Exposure**: Financial Services and Technology form the majority of portfolio holdings in *Figure 9: Sector Allocation Donut*.